# Maternal Health RAG — Document Processing

This notebook prepares trusted maternal health resources from WHO and UNICEF for a Retrieval-Augmented Generation (RAG) system.

## Current Sources

- 8 WHO documents
- 6 UNICEF documents

## Processing Pipeline

PDF Documents → Text Extraction → Cleaning → Chunking → Embeddings → Vector Database

In [1]:
from pathlib import Path
from pypdf import PdfReader

# Define document directories
who_folder = Path("../knowledge_base/WHO")
unicef_folder = Path("../knowledge_base/UNICEF")

# Find all PDF files
who_files = list(who_folder.glob("*.pdf"))
unicef_files = list(unicef_folder.glob("*.pdf"))

print(f"WHO documents: {len(who_files)}")
print(f"UNICEF documents: {len(unicef_files)}")
print(f"Total documents: {len(who_files) + len(unicef_files)}")

WHO documents: 8
UNICEF documents: 6
Total documents: 14


In [2]:
# Select the first WHO PDF for testing

test_file = who_files[0]

print(f"Testing document: {test_file.name}")

Testing document: WHO BLEEDING AFTER BIRTH.pdf


In [3]:
# Open the PDF

reader = PdfReader(test_file)

print(f"Number of pages: {len(reader.pages)}")

Number of pages: 82


In [4]:
# Extract text from the first page

first_page_text = reader.pages[0].extract_text()

print(first_page_text[:3000])

Course on prevention, diagnosis and 
treatment of postpartum haemorrhage
Bleeding after birth 



In [5]:
# Function to extract text from a PDF

def extract_pdf_text(pdf_path):
    """
    Extract text from all pages of a PDF.
    """
    
    reader = PdfReader(pdf_path)
    
    pages = []
    
    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text()
        
        if text:
            pages.append({
                "page_number": page_number,
                "text": text
            })
    
    return pages

In [6]:
# Test the extraction function on the first WHO document

test_pages = extract_pdf_text(who_files[0])

print(f"Document: {who_files[0].name}")
print(f"Pages with extracted text: {len(test_pages)}")

print("\nFirst page:\n")
print(test_pages[0]["text"][:2000])

Document: WHO BLEEDING AFTER BIRTH.pdf
Pages with extracted text: 82

First page:

Course on prevention, diagnosis and 
treatment of postpartum haemorrhage
Bleeding after birth 



In [7]:
# Extract text from all WHO and UNICEF documents

all_documents = []

# Process WHO documents
for pdf_file in who_files:
    
    pages = extract_pdf_text(pdf_file)
    
    for page in pages:
        all_documents.append({
            "organization": "WHO",
            "document": pdf_file.stem,
            "page_number": page["page_number"],
            "text": page["text"]
        })


# Process UNICEF documents
for pdf_file in unicef_files:
    
    pages = extract_pdf_text(pdf_file)
    
    for page in pages:
        all_documents.append({
            "organization": "UNICEF",
            "document": pdf_file.stem,
            "page_number": page["page_number"],
            "text": page["text"]
        })


print(f"Total extracted pages: {len(all_documents)}")

Total extracted pages: 891


In [8]:
# Inspect the first extracted record

print("Organization:", all_documents[0]["organization"])
print("Document:", all_documents[0]["document"])
print("Page:", all_documents[0]["page_number"])

print("\nText:")
print(all_documents[0]["text"][:2000])

Organization: WHO
Document: WHO BLEEDING AFTER BIRTH
Page: 1

Text:
Course on prevention, diagnosis and 
treatment of postpartum haemorrhage
Bleeding after birth 



In [9]:
# Count pages by organization

who_pages = sum(
    1 for document in all_documents
    if document["organization"] == "WHO"
)

unicef_pages = sum(
    1 for document in all_documents
    if document["organization"] == "UNICEF"
)

print(f"WHO pages extracted: {who_pages}")
print(f"UNICEF pages extracted: {unicef_pages}")
print(f"Total pages extracted: {len(all_documents)}")

WHO pages extracted: 612
UNICEF pages extracted: 279
Total pages extracted: 891


In [10]:
import re


def clean_text(text):
    """
    Clean extracted PDF text while preserving the actual content.
    """

    # Replace multiple spaces/tabs with a single space
    text = re.sub(r"[ \t]+", " ", text)

    # Remove excessive blank lines
    text = re.sub(r"\n\s*\n+", "\n\n", text)

    # Remove spaces at the beginning/end of lines
    text = "\n".join(
        line.strip()
        for line in text.splitlines()
    )

    # Remove repeated spaces again after line cleaning
    text = re.sub(r" {2,}", " ", text)

    return text.strip()

In [11]:
# Test cleaning on the first extracted page

original_text = all_documents[0]["text"]

cleaned_text = clean_text(original_text)

print("ORIGINAL TEXT:")
print(original_text[:2000])

print("\n" + "=" * 80 + "\n")

print("CLEANED TEXT:")
print(cleaned_text[:2000])

ORIGINAL TEXT:
Course on prevention, diagnosis and 
treatment of postpartum haemorrhage
Bleeding after birth 



CLEANED TEXT:
Course on prevention, diagnosis and
treatment of postpartum haemorrhage
Bleeding after birth


In [12]:
# Clean the text from all extracted pages

for document in all_documents:
    document["text"] = clean_text(document["text"])

print("Text cleaning completed successfully.")
print(f"Total pages cleaned: {len(all_documents)}")

Text cleaning completed successfully.
Total pages cleaned: 891


In [13]:
# Check for empty text after cleaning

empty_pages = [
    document
    for document in all_documents
    if not document["text"]
]

print(f"Pages with no usable text: {len(empty_pages)}")

Pages with no usable text: 3


In [14]:
# Calculate text length for each page

text_lengths = [
    len(document["text"])
    for document in all_documents
    if document["text"]
]

print(f"Total characters extracted: {sum(text_lengths):,}")
print(f"Average characters per page: {sum(text_lengths) / len(text_lengths):,.0f}")
print(f"Shortest page: {min(text_lengths):,} characters")
print(f"Longest page: {max(text_lengths):,} characters")

Total characters extracted: 2,646,806
Average characters per page: 2,981
Shortest page: 2 characters
Longest page: 38,592 characters


In [15]:
# Chunking
def create_chunks(text, chunk_size=1000, overlap=200):
    """
    Split text into overlapping chunks.

    Parameters:
        text: Cleaned document text
        chunk_size: Maximum number of characters per chunk
        overlap: Number of characters shared between consecutive chunks

    Returns:
        List of text chunks
    """

    if not text:
        return []

    chunks = []

    start = 0
    text_length = len(text)

    while start < text_length:

        end = start + chunk_size

        chunk = text[start:end].strip()

        # Keep only meaningful chunks
        if len(chunk) >= 100:
            chunks.append(chunk)

        # Move forward while maintaining overlap
        start += chunk_size - overlap

    return chunks

In [16]:
# Test chunking on the first page

sample_text = all_documents[0]["text"]

sample_chunks = create_chunks(
    sample_text,
    chunk_size=1000,
    overlap=200
)

print(f"Number of chunks: {len(sample_chunks)}")

for i, chunk in enumerate(sample_chunks[:3], start=1):
    print(f"\n--- Chunk {i} ---\n")
    print(chunk)

Number of chunks: 0


In [17]:
# Find the first page containing at least 500 characters

for document in all_documents:
    if len(document["text"]) >= 500:
        sample_document = document
        break

print("Organization:", sample_document["organization"])
print("Document:", sample_document["document"])
print("Page:", sample_document["page_number"])
print("Text length:", len(sample_document["text"]))

Organization: WHO
Document: WHO BLEEDING AFTER BIRTH
Page: 2
Text length: 4897


In [18]:
# Test chunking on a page with substantial text

sample_text = sample_document["text"]

sample_chunks = create_chunks(
    sample_text,
    chunk_size=1000,
    overlap=200
)

print(f"Number of chunks: {len(sample_chunks)}")

Number of chunks: 6


In [19]:
# Display the first three chunks

for i, chunk in enumerate(sample_chunks[:3], start=1):
    print(f"\n--- Chunk {i} ---\n")
    print(chunk)


--- Chunk 1 ---

Preliminary pages
Bleeding after birth: course on prevention, diagnosis and treatment of postpartum
haemorrhage.
ISBN 978-92-4-011583-5 (print version)
ISBN 978-92-4-011584-2 (electronic version)
© World Health Organization 2025
Some rights reserved. This work is available under the Creative Commons Attribution-
NonCommercial-ShareAlike 3.0 IGO licence (CC BY-NC-SA 3.0 IGO; https://creativecommons.org/
licenses/by-nc-sa/3.0/igo).
Under the terms of this licence, you may copy, redistribute and adapt the work for non-commercial
purposes, provided the work is appropriately cited, as indicated below. In any use of this work,
there should be no suggestion that WHO endorses any specific organization, products or services.
The use of the WHO logo is not permitted. If you adapt the work, then you must license your work
under the same or equivalent Creative Commons licence. If you create a translation of this work,
you should add the following disclaimer along with the suggest

In [20]:
def create_chunks(text, chunk_size=1000, overlap=200):
    """
    Create paragraph-aware overlapping chunks.

    Parameters:
        text: Cleaned document text
        chunk_size: Target maximum number of characters per chunk
        overlap: Approximate overlap between chunks

    Returns:
        List of meaningful text chunks
    """

    if not text:
        return []

    # Split text into paragraphs
    paragraphs = [
        paragraph.strip()
        for paragraph in text.split("\n")
        if paragraph.strip()
    ]

    chunks = []
    current_chunk = ""

    for paragraph in paragraphs:

        # If adding the paragraph keeps the chunk within the target size
        if len(current_chunk) + len(paragraph) + 1 <= chunk_size:
            current_chunk += paragraph + "\n"

        else:
            # Save the current chunk if it is meaningful
            if len(current_chunk.strip()) >= 100:
                chunks.append(current_chunk.strip())

            # Create overlap from the end of the previous chunk
            overlap_text = current_chunk[-overlap:].strip()

            # Start the next chunk
            current_chunk = overlap_text + "\n" + paragraph + "\n"

    # Add the final chunk
    if len(current_chunk.strip()) >= 100:
        chunks.append(current_chunk.strip())

    return chunks

In [21]:
sample_text = sample_document["text"]

sample_chunks = create_chunks(
    sample_text,
    chunk_size=1000,
    overlap=200
)

print(f"Number of chunks: {len(sample_chunks)}")

Number of chunks: 7


In [22]:
for i, chunk in enumerate(sample_chunks[:3], start=1):
    print(f"\n--- Chunk {i} ---\n")
    print(chunk)


--- Chunk 1 ---

Preliminary pages
Bleeding after birth: course on prevention, diagnosis and treatment of postpartum
haemorrhage.
ISBN 978-92-4-011583-5 (print version)
ISBN 978-92-4-011584-2 (electronic version)
© World Health Organization 2025
Some rights reserved. This work is available under the Creative Commons Attribution-
NonCommercial-ShareAlike 3.0 IGO licence (CC BY-NC-SA 3.0 IGO; https://creativecommons.org/
licenses/by-nc-sa/3.0/igo).
Under the terms of this licence, you may copy, redistribute and adapt the work for non-commercial
purposes, provided the work is appropriately cited, as indicated below. In any use of this work,
there should be no suggestion that WHO endorses any specific organization, products or services.
The use of the WHO logo is not permitted. If you adapt the work, then you must license your work
under the same or equivalent Creative Commons licence. If you create a translation of this work,

--- Chunk 2 ---

ices.
The use of the WHO logo is not permitt

In [23]:
def create_chunks(text, chunk_size=1000, overlap=200):
    """
    Create overlapping, paragraph-aware text chunks.

    Parameters:
        text: Cleaned document text
        chunk_size: Target chunk size in characters
        overlap: Approximate overlap between chunks

    Returns:
        List of text chunks
    """

    if not text:
        return []

    # Split text into paragraphs
    paragraphs = [
        paragraph.strip()
        for paragraph in text.split("\n")
        if paragraph.strip()
    ]

    chunks = []
    current_chunk = ""

    for paragraph in paragraphs:

        # Add paragraph if it fits
        if len(current_chunk) + len(paragraph) + 1 <= chunk_size:
            current_chunk += paragraph + "\n"

        else:

            # Save current chunk
            if len(current_chunk.strip()) >= 100:
                chunks.append(current_chunk.strip())

            # Create word-safe overlap
            overlap_text = current_chunk[-overlap:].strip()

            # Remove partial word at the beginning of overlap
            if " " in overlap_text:
                overlap_text = overlap_text.split(" ", 1)[1]

            # Start new chunk
            current_chunk = overlap_text + "\n" + paragraph + "\n"

    # Save final chunk
    if len(current_chunk.strip()) >= 100:
        chunks.append(current_chunk.strip())

    return chunks

In [24]:
sample_chunks = create_chunks(
    sample_document["text"],
    chunk_size=1000,
    overlap=200
)

print(f"Number of chunks: {len(sample_chunks)}")

Number of chunks: 7


In [25]:
for i, chunk in enumerate(sample_chunks[:3], start=1):
    print(f"\n--- Chunk {i} ---\n")
    print(chunk)


--- Chunk 1 ---

Preliminary pages
Bleeding after birth: course on prevention, diagnosis and treatment of postpartum
haemorrhage.
ISBN 978-92-4-011583-5 (print version)
ISBN 978-92-4-011584-2 (electronic version)
© World Health Organization 2025
Some rights reserved. This work is available under the Creative Commons Attribution-
NonCommercial-ShareAlike 3.0 IGO licence (CC BY-NC-SA 3.0 IGO; https://creativecommons.org/
licenses/by-nc-sa/3.0/igo).
Under the terms of this licence, you may copy, redistribute and adapt the work for non-commercial
purposes, provided the work is appropriately cited, as indicated below. In any use of this work,
there should be no suggestion that WHO endorses any specific organization, products or services.
The use of the WHO logo is not permitted. If you adapt the work, then you must license your work
under the same or equivalent Creative Commons licence. If you create a translation of this work,

--- Chunk 2 ---

use of the WHO logo is not permitted. If you

In [26]:
# Find a page with substantial content

for document in all_documents:
    text = document["text"]

    if len(text) >= 2000:
        content_sample = document
        break

print("Organization:", content_sample["organization"])
print("Document:", content_sample["document"])
print("Page:", content_sample["page_number"])
print("Text length:", len(content_sample["text"]))

Organization: WHO
Document: WHO BLEEDING AFTER BIRTH
Page: 2
Text length: 4897


In [27]:
# Create chunks from the content-heavy page

content_chunks = create_chunks(
    content_sample["text"],
    chunk_size=1000,
    overlap=200
)

print(f"Number of chunks: {len(content_chunks)}")

Number of chunks: 7


In [28]:
# Display the first three chunks

for i, chunk in enumerate(content_chunks[:3], start=1):
    print(f"\n--- Chunk {i} ---\n")
    print(chunk)


--- Chunk 1 ---

Preliminary pages
Bleeding after birth: course on prevention, diagnosis and treatment of postpartum
haemorrhage.
ISBN 978-92-4-011583-5 (print version)
ISBN 978-92-4-011584-2 (electronic version)
© World Health Organization 2025
Some rights reserved. This work is available under the Creative Commons Attribution-
NonCommercial-ShareAlike 3.0 IGO licence (CC BY-NC-SA 3.0 IGO; https://creativecommons.org/
licenses/by-nc-sa/3.0/igo).
Under the terms of this licence, you may copy, redistribute and adapt the work for non-commercial
purposes, provided the work is appropriately cited, as indicated below. In any use of this work,
there should be no suggestion that WHO endorses any specific organization, products or services.
The use of the WHO logo is not permitted. If you adapt the work, then you must license your work
under the same or equivalent Creative Commons licence. If you create a translation of this work,

--- Chunk 2 ---

use of the WHO logo is not permitted. If you

In [29]:
# Inspect the first 10 extracted pages

for i, document in enumerate(all_documents[:10]):
    print(
        f"{i}: "
        f"{document['organization']} | "
        f"{document['document']} | "
        f"Page {document['page_number']} | "
        f"{len(document['text'])} characters"
    )

0: WHO | WHO BLEEDING AFTER BIRTH | Page 1 | 92 characters
1: WHO | WHO BLEEDING AFTER BIRTH | Page 2 | 4897 characters
2: WHO | WHO BLEEDING AFTER BIRTH | Page 3 | 4519 characters
3: WHO | WHO BLEEDING AFTER BIRTH | Page 4 | 3525 characters
4: WHO | WHO BLEEDING AFTER BIRTH | Page 5 | 3375 characters
5: WHO | WHO BLEEDING AFTER BIRTH | Page 6 | 32 characters
6: WHO | WHO BLEEDING AFTER BIRTH | Page 7 | 2091 characters
7: WHO | WHO BLEEDING AFTER BIRTH | Page 8 | 135 characters
8: WHO | WHO BLEEDING AFTER BIRTH | Page 9 | 1725 characters
9: WHO | WHO BLEEDING AFTER BIRTH | Page 10 | 1456 characters


In [30]:
# Find pages containing medical-content keywords

medical_keywords = [
    "postpartum haemorrhage",
    "pregnancy",
    "maternal",
    "treatment",
    "diagnosis",
    "bleeding",
    "antenatal"
]

for i, document in enumerate(all_documents):
    
    text_lower = document["text"].lower()
    
    if (
        len(document["text"]) >= 2000
        and any(keyword in text_lower for keyword in medical_keywords)
    ):
        print(
            f"Index: {i} | "
            f"{document['organization']} | "
            f"{document['document']} | "
            f"Page {document['page_number']} | "
            f"{len(document['text'])} characters"
        )
        
        # Stop after finding 5 examples
        if i > 0:
            break

Index: 1 | WHO | WHO BLEEDING AFTER BIRTH | Page 2 | 4897 characters


In [31]:
# Select the medical-content page

content_sample = all_documents[1]

print("Organization:", content_sample["organization"])
print("Document:", content_sample["document"])
print("Page:", content_sample["page_number"])
print("Text length:", len(content_sample["text"]))

Organization: WHO
Document: WHO BLEEDING AFTER BIRTH
Page: 2
Text length: 4897


In [32]:
# Display the beginning of the selected page

print(content_sample["text"][:3000])

Preliminary pages
Bleeding after birth: course on prevention, diagnosis and treatment of postpartum
haemorrhage.
ISBN 978-92-4-011583-5 (print version)
ISBN 978-92-4-011584-2 (electronic version)
© World Health Organization 2025
Some rights reserved. This work is available under the Creative Commons Attribution-
NonCommercial-ShareAlike 3.0 IGO licence (CC BY-NC-SA 3.0 IGO; https://creativecommons.org/
licenses/by-nc-sa/3.0/igo).
Under the terms of this licence, you may copy, redistribute and adapt the work for non-commercial
purposes, provided the work is appropriately cited, as indicated below. In any use of this work,
there should be no suggestion that WHO endorses any specific organization, products or services.
The use of the WHO logo is not permitted. If you adapt the work, then you must license your work
under the same or equivalent Creative Commons licence. If you create a translation of this work,
you should add the following disclaimer along with the suggested citation: “This

In [33]:
# Find a likely content page after the preliminary pages

for i, document in enumerate(all_documents):
    
    # Skip the first 20 extracted pages
    if i < 20:
        continue

    text = document["text"]

    if len(text) >= 2000:
        print(
            f"Index: {i} | "
            f"Organization: {document['organization']} | "
            f"Document: {document['document']} | "
            f"Page: {document['page_number']} | "
            f"Characters: {len(text)}"
        )
        
        break

Index: 20 | Organization: WHO | Document: WHO BLEEDING AFTER BIRTH | Page: 21 | Characters: 3587


In [34]:
# Select the first substantial page after the preliminary section

content_sample = None

for i, document in enumerate(all_documents):
    
    if i < 20:
        continue
    
    if len(document["text"]) >= 2000:
        content_sample = document
        break


print("Organization:", content_sample["organization"])
print("Document:", content_sample["document"])
print("Page:", content_sample["page_number"])
print("Text length:", len(content_sample["text"]))

Organization: WHO
Document: WHO BLEEDING AFTER BIRTH
Page: 21
Text length: 3587


In [35]:
# Display the selected page

print(content_sample["text"][:4000])

Module 2: Prevention and early diagnosis of PPH Start measuring blood loss
Explain
• Measure blood loss objectively to
quickly diagnose and treat PPH.
• A blood loss measurement tool is
recommended for all women, both for
vaginal and caesarean births.
Blood loss measurement tools
• For vaginal birth, use a tool that:
— easily shows volume of blood loss
— collects all blood and avoids spillage
— ensures the woman’s comfort and
choice of birth position.
• Examples:
— drapes with a calibrated funnel
— trays with calibrated lines
— other innovations.
Place the tool
• Immediately after the baby is born and
you have given the uterotonic.
• For non-supine position birth:
— assist her to a supine position
— place baby on her chest
— give uterotonic, then place tool.
Measure blood loss accurately
• Sweep blood and clots into the tool.
• If using a drape:
— hang the funnel over the edge of the
bed, or
— lift the funnel so blood collects at the
bottom.
• Check continuously to diagnose PPH
quickly

In [36]:
# Test chunking on the selected medical-content page

content_chunks = create_chunks(
    content_sample["text"],
    chunk_size=1000,
    overlap=200
)

print(f"Number of chunks: {len(content_chunks)}")

Number of chunks: 5


In [37]:
# Display the first three chunks

for i, chunk in enumerate(content_chunks[:3], start=1):
    print(f"\n--- Chunk {i} ---\n")
    print(chunk)


--- Chunk 1 ---

Module 2: Prevention and early diagnosis of PPH Start measuring blood loss
Explain
• Measure blood loss objectively to
quickly diagnose and treat PPH.
• A blood loss measurement tool is
recommended for all women, both for
vaginal and caesarean births.
Blood loss measurement tools
• For vaginal birth, use a tool that:
— easily shows volume of blood loss
— collects all blood and avoids spillage
— ensures the woman’s comfort and
choice of birth position.
• Examples:
— drapes with a calibrated funnel
— trays with calibrated lines
— other innovations.
Place the tool
• Immediately after the baby is born and
you have given the uterotonic.
• For non-supine position birth:
— assist her to a supine position
— place baby on her chest
— give uterotonic, then place tool.
Measure blood loss accurately
• Sweep blood and clots into the tool.
• If using a drape:
— hang the funnel over the edge of the
bed, or
— lift the funnel so blood collects at the
bottom.
• Check continuously to di

In [38]:
import polars as pl

# Path to the source information
sources_path = Path("../knowledge_base/sources.csv")

# Load sources using Polars
sources_df = pl.read_csv(sources_path)

# Display the sources
print(f"Total sources in sources.csv: {sources_df.height}")

sources_df

Total sources in sources.csv: 30


id,organization,title,type,url
i64,str,str,str,str
1,"""WHO""","""WHO Recommendations on Antenat…","""PDF""","""https://www.who.int/publicatio…"
2,"""WHO""","""WHO Antenatal Care Executive S…","""PDF""","""https://www.who.int/publicatio…"
3,"""WHO""","""WHO Antenatal Care Highlights …","""PDF""","""https://www.who.int/publicatio…"
4,"""WHO""","""Pregnancy Childbirth Postpartu…","""PDF""","""https://www.who.int/publicatio…"
5,"""WHO""","""WHO Recommendations for Preven…","""PDF""","""https://www.who.int/publicatio…"
…,…,…,…,…
26,"""UNICEF Nigeria""","""Safe and Healthy Pregnancy""","""WEBPAGE""","""https://www.unicef.org/nigeria…"
27,"""UNICEF""","""Programme Guidance on Maternal…","""PDF""","""https://www.unicef.org/documen…"
28,"""UNICEF""","""Nutrition Commodities for Wome…","""PDF""","""https://www.unicef.org/documen…"


In [39]:
# Select only PDF resources
pdf_sources = sources_df.filter(
    pl.col("type") == "PDF"
)

print(f"PDF sources: {pdf_sources.height}")

PDF sources: 14


In [40]:
# Display the PDF sources

print(
    pdf_sources.select(
        ["id", "organization", "title", "url"]
    )
)

shape: (14, 4)
┌─────┬────────────────┬─────────────────────────────────┬─────────────────────────────────┐
│ id  ┆ organization   ┆ title                           ┆ url                             │
│ --- ┆ ---            ┆ ---                             ┆ ---                             │
│ i64 ┆ str            ┆ str                             ┆ str                             │
╞═════╪════════════════╪═════════════════════════════════╪═════════════════════════════════╡
│ 1   ┆ WHO            ┆ WHO Recommendations on Antenat… ┆ https://www.who.int/publicatio… │
│ 2   ┆ WHO            ┆ WHO Antenatal Care Executive S… ┆ https://www.who.int/publicatio… │
│ 3   ┆ WHO            ┆ WHO Antenatal Care Highlights … ┆ https://www.who.int/publicatio… │
│ 4   ┆ WHO            ┆ Pregnancy Childbirth Postpartu… ┆ https://www.who.int/publicatio… │
│ 5   ┆ WHO            ┆ WHO Recommendations for Preven… ┆ https://www.who.int/publicatio… │
│ …   ┆ …              ┆ …                             

In [41]:
# Create a simple lookup dictionary for PDF sources

pdf_source_records = pdf_sources.to_dicts()

print(f"Number of PDF source records: {len(pdf_source_records)}")

Number of PDF source records: 14


In [42]:
# Display all unique documents currently extracted

unique_documents = {}

for document in all_documents:
    key = (
        document["organization"],
        document["document"]
    )
    
    unique_documents[key] = True


for organization, document_name in unique_documents:
    print(f"{organization} | {document_name}")

WHO | WHO BLEEDING AFTER BIRTH
WHO | WHO POLICY PREECLAMPSIA
WHO | Who pregnancy, childbirth, postpartum
WHO | WHO recommendation on the loss of postpartum blood
WHO | WHO recommendation on tranexamic acid for the treatment of postpartum haemorrhage
WHO | Who recommendations for prevention and treatment of pre-eclampsia and eclampsia
WHO | Who recommendations on Antenatal care for positive pregnancy experience
WHO | WHO-RHR-18.02-eng
UNICEF | UNICEF Maternal Nutrition Counselling Brief
UNICEF | UNICEF Maternal Nutrition Programming Guidance
UNICEF | UNICEF Nigeria-evaluation-of-maternal-newborn-and-child-health-week_0.pdf
UNICEF | UNICEF PBWG-Humanitarian-Evidence-review-final
UNICEF | UNICEF Technical-Bulletin-No26-Nutrition-Products-for-Pregnancy-2021
UNICEF | UNICEF Women's nutrition humanitarian guidance-2024-FINAL-Sept2024.pdf


In [73]:
# Mapping between extracted document names and sources.csv IDs

document_source_map = {
    # WHO documents
    "WHO BLEEDING AFTER BIRTH": 10,
    "WHO POLICY PREECLAMPSIA": 6,
    "Who pregnancy, childbirth, postpartum": 4,
    "WHO recommendation on the loss of postpartum blood": 8,
    "WHO recommendation on tranexamic acid for the treatment of postpartum haemorrhage": 9,
    "Who recommendations for prevention and treatment of pre-eclampsia and eclampsia": 5,
    "Who recommendations on Antenatal care for positive pregnancy experience": 1,
    "WHO-RHR-18.02-eng": 3,

    # UNICEF documents
    "UNICEF Maternal Nutrition Counselling Brief": 31,
    "UNICEF Maternal Nutrition Programming Guidance": 27,
    "UNICEF Nigeria-evaluation-of-maternal-newborn-and-child-health-week_0.pdf": 30,
    "UNICEF PBWG-Humanitarian-Evidence-review-final": 32,
    "UNICEF Technical-Bulletin-No26-Nutrition-Products-for-Pregnancy-2021": 28,
    "UNICEF Women's nutrition humanitarian guidance-2024-FINAL-Sept2024.pdf": 29,
}

print(f"Total document mappings: {len(document_source_map)}")

Total document mappings: 14


In [44]:
# Inspect the structure of one extracted document

print(all_documents[0].keys())

dict_keys(['organization', 'document', 'page_number', 'text'])


In [45]:
# Show the metadata of the first extracted document

print(all_documents[0])

{'organization': 'WHO', 'document': 'WHO BLEEDING AFTER BIRTH', 'page_number': 1, 'text': 'Course on prevention, diagnosis and\ntreatment of postpartum haemorrhage\nBleeding after birth'}


In [46]:
# Keep only the information we need from sources.csv

source_lookup = sources_df.select([
    "id",
    "organization",
    "title",
    "url"
])

print(f"Source records available: {source_lookup.height}")

Source records available: 30


In [47]:
# Mapping from extracted document names to sources.csv IDs

document_source_map = {
    "WHO BLEEDING AFTER BIRTH": 10,
    "WHO POLICY PREECLAMPSIA": 6,
    "Who pregnancy, childbirth, postpartum": 4,
    "WHO recommendation on the loss of postpartum blood": 8,
    "WHO recommendation on tranexamic acid for the treatment of postpartum haemorrhage": 9,
    "Who recommendations for prevention and treatment of pre-eclampsia and eclampsia": 5,
    "Who recommendations on Antenatal care for positive pregnancy experience": 1,
    "WHO-RHR-18.02-eng": 3,

    "UNICEF Maternal Nutrition Programming Guidance": 27,
    "UNICEF Nigeria-evaluation-of-maternal-newborn-and-child-health-week_0.pdf": 30,
    "UNICEF Technical-Bulletin-No26-Nutrition-Products-for-Pregnancy-2021": 28,
    "UNICEF Women's nutrition humanitarian guidance-2024-FINAL-Sept2024.pdf": 29,
}

print(f"Confirmed document mappings: {len(document_source_map)}")

Confirmed document mappings: 12


In [48]:
# Find documents that are not yet mapped

unique_documents = set(
    document["document"]
    for document in all_documents
)

unmatched_documents = [
    document
    for document in unique_documents
    if document not in document_source_map
]

print("Unmatched documents:")

for document in unmatched_documents:
    print("-", document)

Unmatched documents:
- UNICEF Maternal Nutrition Counselling Brief
- UNICEF PBWG-Humanitarian-Evidence-review-final


In [49]:
def create_chunks(text, chunk_size=1000, overlap=200):
    """
    Create overlapping, paragraph-aware text chunks.

    Parameters:
        text: Cleaned document text
        chunk_size: Target chunk size in characters
        overlap: Approximate overlap between chunks

    Returns:
        List of text chunks
    """

    if not text:
        return []

    paragraphs = [
        paragraph.strip()
        for paragraph in text.split("\n")
        if paragraph.strip()
    ]

    chunks = []
    current_chunk = ""

    for paragraph in paragraphs:

        if len(current_chunk) + len(paragraph) + 1 <= chunk_size:
            current_chunk += paragraph + "\n"

        else:

            if len(current_chunk.strip()) >= 100:
                chunks.append(current_chunk.strip())

            overlap_text = current_chunk[-overlap:].strip()

            if " " in overlap_text:
                overlap_text = overlap_text.split(" ", 1)[1]

            current_chunk = overlap_text + "\n" + paragraph + "\n"

    if len(current_chunk.strip()) >= 100:
        chunks.append(current_chunk.strip())

    return chunks

In [50]:
# Test the complete chunking process on one document

test_document = all_documents[0]

test_chunks = create_chunks(
    test_document["text"],
    chunk_size=1000,
    overlap=200
)

print("Organization:", test_document["organization"])
print("Document:", test_document["document"])
print("Page:", test_document["page_number"])
print("Number of chunks:", len(test_chunks))

Organization: WHO
Document: WHO BLEEDING AFTER BIRTH
Page: 1
Number of chunks: 0


In [52]:
# Find a page with enough text to create a meaningful chunk

test_document = None

for document in all_documents:
    if len(document["text"]) >= 1000:
        test_document = document
        break

print("Organization:", test_document["organization"])
print("Document:", test_document["document"])
print("Page:", test_document["page_number"])
print("Text length:", len(test_document["text"]))

Organization: WHO
Document: WHO BLEEDING AFTER BIRTH
Page: 2
Text length: 4897


In [53]:
# Create chunks from the selected page

test_chunks = create_chunks(
    test_document["text"],
    chunk_size=1000,
    overlap=200
)

print("Number of chunks:", len(test_chunks))

Number of chunks: 7


In [54]:
# Display the first chunk

print("\n--- TEST CHUNK ---\n")
print(test_chunks[0])


--- TEST CHUNK ---

Preliminary pages
Bleeding after birth: course on prevention, diagnosis and treatment of postpartum
haemorrhage.
ISBN 978-92-4-011583-5 (print version)
ISBN 978-92-4-011584-2 (electronic version)
© World Health Organization 2025
Some rights reserved. This work is available under the Creative Commons Attribution-
NonCommercial-ShareAlike 3.0 IGO licence (CC BY-NC-SA 3.0 IGO; https://creativecommons.org/
licenses/by-nc-sa/3.0/igo).
Under the terms of this licence, you may copy, redistribute and adapt the work for non-commercial
purposes, provided the work is appropriately cited, as indicated below. In any use of this work,
there should be no suggestion that WHO endorses any specific organization, products or services.
The use of the WHO logo is not permitted. If you adapt the work, then you must license your work
under the same or equivalent Creative Commons licence. If you create a translation of this work,


In [55]:
# Find a page containing actual medical content

for i, document in enumerate(all_documents):
    text = document["text"].lower()

    if (
        "module 2" in text
        and "prevention and early diagnosis" in text
    ):
        print(
            f"Index: {i} | "
            f"Organization: {document['organization']} | "
            f"Document: {document['document']} | "
            f"Page: {document['page_number']} | "
            f"Characters: {len(document['text'])}"
        )
        break

Index: 16 | Organization: WHO | Document: WHO BLEEDING AFTER BIRTH | Page: 17 | Characters: 3212


In [56]:
# Select the actual medical-content page

content_document = None

for document in all_documents:
    text = document["text"].lower()

    if (
        "module 2" in text
        and "prevention and early diagnosis" in text
    ):
        content_document = document
        break

print("Organization:", content_document["organization"])
print("Document:", content_document["document"])
print("Page:", content_document["page_number"])
print("Text length:", len(content_document["text"]))

Organization: WHO
Document: WHO BLEEDING AFTER BIRTH
Page: 17
Text length: 3212


In [57]:
# Create chunks from the actual medical-content page

content_chunks = create_chunks(
    content_document["text"],
    chunk_size=1000,
    overlap=200
)

print("Number of chunks:", len(content_chunks))

Number of chunks: 4


In [58]:
# Display the first chunk

print("\n--- MEDICAL CONTENT CHUNK ---\n")
print(content_chunks[0])


--- MEDICAL CONTENT CHUNK ---

Module 2: Prevention and early diagnosis of PPH
Explain
Prevent PPH
Actively preventing PPH ensures:
— a shorter third stage of labour
— less blood loss
— fewer cases of PPH.
Steps to prevent and identify PPH
1. Give a uterotonic to prevent PPH in
both vaginal and caesarean births after
the last baby is born.
2. Place the blood measurement tool
and monitor blood loss closely.
3. Deliver placenta with controlled
cord traction (CCT) aft
er clamping and
cutting the cord.
4. Check uterine tone and massage if soft.
Once the woman is stable
• Check the placenta for completeness.
• Check for and repair tears.
• Assess and record tone, bleeding and
vital signs.
• Keep the blood loss monitoring tool in
place for the first hour for all women. If
a woman continues to bleed, leave the
tool in place for another hour.
Continue care for woman and baby
The time after birth is critical for both the
woman and her baby.
• Ensure the baby is breathing well and
kept warm.


In [59]:
# Create chunks for all extracted pages

all_chunks = []

for document in all_documents:

    chunks = create_chunks(
        document["text"],
        chunk_size=1000,
        overlap=200
    )

    for chunk_number, chunk in enumerate(chunks, start=1):

        all_chunks.append({
            "organization": document["organization"],
            "document": document["document"],
            "page_number": document["page_number"],
            "chunk_number": chunk_number,
            "text": chunk
        })

print(f"Total chunks created: {len(all_chunks)}")

Total chunks created: 3596


In [60]:
import polars as pl

chunks_df = pl.DataFrame(all_chunks)

print(f"Total chunks: {chunks_df.height}")
print(f"Columns: {chunks_df.columns}")

Total chunks: 3596
Columns: ['organization', 'document', 'page_number', 'chunk_number', 'text']


In [61]:
chunks_df.head(5)

organization,document,page_number,chunk_number,text
str,str,i64,i64,str
"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,1,"""Preliminary pages Bleeding aft…"
"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,2,"""use of the WHO logo is not per…"
"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,3,"""of postpartum haemorrhage. Gen…"
"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,4,"""with the user. General disclai…"
"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,5,"""initial capital letters. All r…"


In [62]:
chunks_df.group_by("organization").agg(
    pl.len().alias("number_of_chunks")
).sort("organization")

organization,number_of_chunks
str,u32
"""UNICEF""",1020
"""WHO""",2576


In [63]:
chunks_df.group_by(
    ["organization", "document"]
).agg(
    pl.len().alias("number_of_chunks")
).sort(
    ["organization", "document"]
)

organization,document,number_of_chunks
str,str,u32
"""UNICEF""","""UNICEF Maternal Nutrition Coun…",78
"""UNICEF""","""UNICEF Maternal Nutrition Prog…",214
"""UNICEF""","""UNICEF Nigeria-evaluation-of-m…",332
"""UNICEF""","""UNICEF PBWG-Humanitarian-Evide…",198
"""UNICEF""","""UNICEF Technical-Bulletin-No26…",9
…,…,…
"""WHO""","""WHO recommendation on tranexam…",139
"""WHO""","""WHO-RHR-18.02-eng""",63
"""WHO""","""Who pregnancy, childbirth, pos…",927


In [64]:
# Check the chunk counts for every document

document_chunk_counts = (
    chunks_df
    .group_by(["organization", "document"])
    .agg(
        pl.len().alias("number_of_chunks")
    )
    .sort(["organization", "number_of_chunks"])
)

document_chunk_counts

organization,document,number_of_chunks
str,str,u32
"""UNICEF""","""UNICEF Technical-Bulletin-No26…",9
"""UNICEF""","""UNICEF Maternal Nutrition Coun…",78
"""UNICEF""","""UNICEF Women's nutrition human…",189
"""UNICEF""","""UNICEF PBWG-Humanitarian-Evide…",198
"""UNICEF""","""UNICEF Maternal Nutrition Prog…",214
…,…,…
"""WHO""","""WHO recommendation on the loss…",144
"""WHO""","""Who recommendations for preven…",179
"""WHO""","""WHO BLEEDING AFTER BIRTH""",245


In [65]:
# Check the shortest and longest chunks

chunk_lengths = (
    chunks_df
    .with_columns(
        pl.col("text")
        .str.len_chars()
        .alias("chunk_length")
    )
)

print("Shortest chunk:", chunk_lengths["chunk_length"].min())
print("Longest chunk:", chunk_lengths["chunk_length"].max())
print("Average chunk:", round(chunk_lengths["chunk_length"].mean(), 2))

Shortest chunk: 100
Longest chunk: 999
Average chunk: 880.57


In [66]:
# Inspect the chunks from the document with only 9 chunks

chunks_df.filter(
    pl.col("document") == "UNICEF Technical-Bulletin-No26-Nutrition-Products-for-Pregnancy-2021"
).select(
    ["page_number", "chunk_number", "text"]
)

page_number,chunk_number,text
i64,i64,str
1,1,"""BACKGROUND The UNICEF Strategi…"
1,2,"""the absence of adequate nutrit…"
1,3,"""for pregnant women as part of …"
1,4,"""women and newborns. To support…"
1,5,"""Reduces the risk of stillbirth…"
2,1,"""Nutrient values of Multiple Mi…"
2,2,"""refer to the UNICEF Supply Cat…"
2,3,"""antenatal care recommendations…"
2,4,"""- lampsia and its complication…"


In [67]:
# Create a unique ID for every chunk

chunks_df = chunks_df.with_row_index(
    name="chunk_id",
    offset=1
)

print(f"Total chunks: {chunks_df.height}")

chunks_df.head(5)

Total chunks: 3596


chunk_id,organization,document,page_number,chunk_number,text
u32,str,str,i64,i64,str
1,"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,1,"""Preliminary pages Bleeding aft…"
2,"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,2,"""use of the WHO logo is not per…"
3,"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,3,"""of postpartum haemorrhage. Gen…"
4,"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,4,"""with the user. General disclai…"
5,"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,5,"""initial capital letters. All r…"


In [68]:
# Check the first and last chunk IDs

print("First chunk ID:", chunks_df["chunk_id"].min())
print("Last chunk ID:", chunks_df["chunk_id"].max())

First chunk ID: 1
Last chunk ID: 3596


In [69]:
# Find the two documents without confirmed source mappings

unmatched_documents = [
    document
    for document in chunks_df["document"].unique().to_list()
    if document not in document_source_map
]

print("Documents without confirmed source mapping:")

for document in unmatched_documents:
    print("-", document)

Documents without confirmed source mapping:
- UNICEF PBWG-Humanitarian-Evidence-review-final
- UNICEF Maternal Nutrition Counselling Brief


In [70]:
# Count chunks belonging to the unmatched documents

unmatched_chunk_count = chunks_df.filter(
    pl.col("document").is_in(unmatched_documents)
).height

print(f"Chunks without confirmed source URL: {unmatched_chunk_count}")

Chunks without confirmed source URL: 276


In [71]:
# Inspect the two documents without confirmed source URLs

unmatched_details = (
    chunks_df
    .filter(
        pl.col("document").is_in(unmatched_documents)
    )
    .group_by(["organization", "document"])
    .agg(
        pl.len().alias("number_of_chunks"),
        pl.col("page_number").min().alias("first_page"),
        pl.col("page_number").max().alias("last_page")
    )
    .sort("document")
)

unmatched_details

organization,document,number_of_chunks,first_page,last_page
str,str,u32,i64,i64
"""UNICEF""","""UNICEF Maternal Nutrition Coun…",78,1,16
"""UNICEF""","""UNICEF PBWG-Humanitarian-Evide…",198,1,39


In [72]:
# Display the first chunk from each unmatched document

for document in unmatched_documents:

    print("\n" + "=" * 80)
    print(document)
    print("=" * 80)

    first_chunk = (
        chunks_df
        .filter(pl.col("document") == document)
        .sort(["page_number", "chunk_number"])
        .select("text")
        .head(1)
    )

    print(first_chunk["text"][0][:2000])


UNICEF PBWG-Humanitarian-Evidence-review-final
Evidence Review to inform the Programme Guidance to Protect the Nutrition of Women and Adolescent Girls in Humanitarian Settings | A
Evidence Review
to Inform the Programme
Guidance to Protect the Nutrition
of Women and Adolescent Girls in
Humanitarian Settings

UNICEF Maternal Nutrition Counselling Brief
1
Counselling to improve maternal nutrition
COUNSELLING TO IMPROVE
MATERNAL NUTRITION
Considerations for programming
with quality, equity and scale
A TECHNICAL BRIEF


In [74]:
# Check for documents without a source mapping

unmatched_documents = [
    document
    for document in chunks_df["document"].unique().to_list()
    if document not in document_source_map
]

print("Unmatched documents:")

if unmatched_documents:
    for document in unmatched_documents:
        print("-", document)
else:
    print("None — all 14 documents have a source mapping.")

Unmatched documents:
None — all 14 documents have a source mapping.


In [75]:
# Create a Polars DataFrame from the document-to-source mapping

mapping_rows = [
    {
        "document": document,
        "source_id": source_id
    }
    for document, source_id in document_source_map.items()
]

mapping_df = pl.DataFrame(mapping_rows)

print(f"Document mappings: {mapping_df.height}")

mapping_df

Document mappings: 14


document,source_id
str,i64
"""WHO BLEEDING AFTER BIRTH""",10
"""WHO POLICY PREECLAMPSIA""",6
"""Who pregnancy, childbirth, pos…",4
"""WHO recommendation on the loss…",8
"""WHO recommendation on tranexam…",9
…,…
"""UNICEF Maternal Nutrition Prog…",27
"""UNICEF Nigeria-evaluation-of-m…",30
"""UNICEF PBWG-Humanitarian-Evide…",32


In [76]:
# Attach source IDs to every chunk

chunks_with_source_id = chunks_df.join(
    mapping_df,
    on="document",
    how="left"
)

print(f"Total chunks: {chunks_with_source_id.height}")

chunks_with_source_id.head(5)

Total chunks: 3596


chunk_id,organization,document,page_number,chunk_number,text,source_id
u32,str,str,i64,i64,str,i64
1,"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,1,"""Preliminary pages Bleeding aft…",10
2,"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,2,"""use of the WHO logo is not per…",10
3,"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,3,"""of postpartum haemorrhage. Gen…",10
4,"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,4,"""with the user. General disclai…",10
5,"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,5,"""initial capital letters. All r…",10


In [77]:
# Check for chunks without a source ID

missing_source_ids = chunks_with_source_id.filter(
    pl.col("source_id").is_null()
).height

print(f"Chunks without source ID: {missing_source_ids}")

Chunks without source ID: 0


In [78]:
# Select the source information we need

source_metadata = sources_df.select([
    "id",
    "title",
    "url"
]).rename({
    "id": "source_id"
})

In [79]:
# Attach source title and URL to every chunk

chunks_with_metadata = chunks_with_source_id.join(
    source_metadata,
    on="source_id",
    how="left"
)

print(f"Total chunks: {chunks_with_metadata.height}")

Total chunks: 3596


In [80]:
print(chunks_with_metadata.columns)

['chunk_id', 'organization', 'document', 'page_number', 'chunk_number', 'text', 'source_id', 'title', 'url']


In [81]:
chunks_with_metadata.select([
    "chunk_id",
    "organization",
    "document",
    "page_number",
    "chunk_number",
    "title",
    "url",
    "text"
]).head(3)

chunk_id,organization,document,page_number,chunk_number,title,url,text
u32,str,str,i64,i64,str,str,str
1,"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,1,"""Bleeding After Birth: Course o…","""https://www.who.int/publicatio…","""Preliminary pages Bleeding aft…"
2,"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,2,"""Bleeding After Birth: Course o…","""https://www.who.int/publicatio…","""use of the WHO logo is not per…"
3,"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,3,"""Bleeding After Birth: Course o…","""https://www.who.int/publicatio…","""of postpartum haemorrhage. Gen…"


In [82]:
# Find chunks containing common preliminary-page terms

preliminary_keywords = [
    "ISBN",
    "© World Health Organization",
    "Some rights reserved",
    "Creative Commons",
    "General disclaimers",
    "Suggested citation"
]

preliminary_chunks = chunks_with_metadata.filter(
    pl.any_horizontal([
        pl.col("text").str.contains(keyword, literal=True)
        for keyword in preliminary_keywords
    ])
)

print(f"Potential preliminary chunks: {preliminary_chunks.height}")

Potential preliminary chunks: 97


In [83]:
# Check where the potential preliminary chunks come from

preliminary_by_document = (
    preliminary_chunks
    .group_by(["organization", "document"])
    .agg(
        pl.len().alias("flagged_chunks")
    )
    .sort("flagged_chunks", descending=True)
)

preliminary_by_document

organization,document,flagged_chunks
str,str,u32
"""WHO""","""WHO BLEEDING AFTER BIRTH""",72
"""WHO""","""WHO POLICY PREECLAMPSIA""",5
"""WHO""","""WHO recommendation on tranexam…",4
"""WHO""","""WHO recommendation on the loss…",4
"""UNICEF""","""UNICEF Maternal Nutrition Prog…",2
…,…,…
"""WHO""","""Who recommendations for preven…",2
"""UNICEF""","""UNICEF Women's nutrition human…",1
"""UNICEF""","""UNICEF PBWG-Humanitarian-Evide…",1


In [84]:
# Inspect the first 10 potential preliminary chunks

preliminary_chunks.select([
    "organization",
    "document",
    "page_number",
    "chunk_number",
    "text"
]).head(10)

organization,document,page_number,chunk_number,text
str,str,i64,i64,str
"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,1,"""Preliminary pages Bleeding aft…"
"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,2,"""use of the WHO logo is not per…"
"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,3,"""of postpartum haemorrhage. Gen…"
"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,4,"""with the user. General disclai…"
"""WHO""","""WHO BLEEDING AFTER BIRTH""",10,2,"""bleeding Prepare for birth Che…"
"""WHO""","""WHO BLEEDING AFTER BIRTH""",11,4,"""bleeding Prepare for birth Che…"
"""WHO""","""WHO BLEEDING AFTER BIRTH""",12,2,"""min Out Placenta out? Check to…"
"""WHO""","""WHO BLEEDING AFTER BIRTH""",13,4,"""infection Provide respectful c…"
"""WHO""","""WHO BLEEDING AFTER BIRTH""",14,2,"""bleeding Prepare for birth Che…"


In [85]:
# Look specifically for administrative/copyright content

administrative_patterns = [
    "ISBN",
    "© World Health Organization",
    "Some rights reserved",
    "Creative Commons",
    "The use of the WHO logo is not permitted",
    "General disclaimers",
    "Suggested citation"
]

administrative_chunks = preliminary_chunks.filter(
    pl.any_horizontal([
        pl.col("text").str.contains(pattern, literal=True)
        for pattern in administrative_patterns
    ])
)

print(f"Administrative chunks identified: {administrative_chunks.height}")

Administrative chunks identified: 97


In [86]:
# Inspect the administrative chunks

administrative_chunks.select([
    "organization",
    "document",
    "page_number",
    "chunk_number",
    "text"
])

organization,document,page_number,chunk_number,text
str,str,i64,i64,str
"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,1,"""Preliminary pages Bleeding aft…"
"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,2,"""use of the WHO logo is not per…"
"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,3,"""of postpartum haemorrhage. Gen…"
"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,4,"""with the user. General disclai…"
"""WHO""","""WHO BLEEDING AFTER BIRTH""",10,2,"""bleeding Prepare for birth Che…"
…,…,…,…,…
"""UNICEF""","""UNICEF Maternal Nutrition Coun…",2,1,"""2 Counselling to improve mater…"
"""UNICEF""","""UNICEF Maternal Nutrition Prog…",2,1,"""2 UNICEF Programming Guidance …"
"""UNICEF""","""UNICEF Maternal Nutrition Prog…",2,2,"""within are those of the author…"


In [87]:
# Inspect all chunks from page 2 of the WHO Bleeding After Birth document

chunks_with_metadata.filter(
    (pl.col("document") == "WHO BLEEDING AFTER BIRTH") &
    (pl.col("page_number") == 2)
).select([
    "page_number",
    "chunk_number",
    "text"
])

page_number,chunk_number,text
i64,i64,str
2,1,"""Preliminary pages Bleeding aft…"
2,2,"""use of the WHO logo is not per…"
2,3,"""of postpartum haemorrhage. Gen…"
2,4,"""with the user. General disclai…"
2,5,"""initial capital letters. All r…"
2,6,"""a uterotonic within 1 min 8 St…"
2,7,"""clots/debris 31 Start second I…"


In [88]:
# Remove only clearly administrative/copyright chunks
# Keep all medical and educational content.

administrative_phrases = [
    "ISBN",
    "© World Health Organization",
    "Some rights reserved",
    "Creative Commons",
    "The use of the WHO logo is not permitted",
    "General disclaimers",
    "Suggested citation"
]

is_administrative = pl.any_horizontal([
    pl.col("text").str.contains(
        phrase,
        literal=True
    )
    for phrase in administrative_phrases
])

final_chunks_df = chunks_with_metadata.filter(
    ~is_administrative
)

print("Original chunks:", chunks_with_metadata.height)
print("Final chunks:", final_chunks_df.height)
print("Removed chunks:", chunks_with_metadata.height - final_chunks_df.height)

Original chunks: 3596
Final chunks: 3499
Removed chunks: 97


In [89]:
# Inspect the chunks that would be removed

removed_chunks = chunks_with_metadata.filter(
    is_administrative
)

removed_chunks.select([
    "organization",
    "document",
    "page_number",
    "chunk_number",
    "text"
])

organization,document,page_number,chunk_number,text
str,str,i64,i64,str
"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,1,"""Preliminary pages Bleeding aft…"
"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,2,"""use of the WHO logo is not per…"
"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,3,"""of postpartum haemorrhage. Gen…"
"""WHO""","""WHO BLEEDING AFTER BIRTH""",2,4,"""with the user. General disclai…"
"""WHO""","""WHO BLEEDING AFTER BIRTH""",10,2,"""bleeding Prepare for birth Che…"
…,…,…,…,…
"""UNICEF""","""UNICEF Maternal Nutrition Coun…",2,1,"""2 Counselling to improve mater…"
"""UNICEF""","""UNICEF Maternal Nutrition Prog…",2,1,"""2 UNICEF Programming Guidance …"
"""UNICEF""","""UNICEF Maternal Nutrition Prog…",2,2,"""within are those of the author…"


In [90]:
# Keep the complete chunk dataset.
# We will not remove chunks based on broad keywords.

final_chunks_df = chunks_with_metadata.clone()

print("Final chunks:", final_chunks_df.height)
print("Expected chunks: 3596")

Final chunks: 3596
Expected chunks: 3596


In [91]:
# Check for missing values in important metadata columns

metadata_columns = [
    "chunk_id",
    "organization",
    "document",
    "page_number",
    "chunk_number",
    "source_id",
    "title",
    "url",
    "text"
]

for column in metadata_columns:
    missing = final_chunks_df.filter(
        pl.col(column).is_null()
    ).height

    print(f"{column}: {missing} missing")

chunk_id: 0 missing
organization: 0 missing
document: 0 missing
page_number: 0 missing
chunk_number: 0 missing
source_id: 0 missing
title: 276 missing
url: 276 missing
text: 0 missing


In [92]:
# Check the source IDs for the 276 chunks missing title and URL

missing_metadata = final_chunks_df.filter(
    pl.col("title").is_null() | pl.col("url").is_null()
)

missing_metadata.select([
    "organization",
    "document",
    "source_id"
]).unique()

organization,document,source_id
str,str,i64
"""UNICEF""","""UNICEF PBWG-Humanitarian-Evide…",32
"""UNICEF""","""UNICEF Maternal Nutrition Coun…",31


In [93]:
# Check whether sources_df contains source IDs 31 and 32

sources_df.filter(
    pl.col("id").is_in([31, 32])
)

id,organization,title,type,url
i64,str,str,str,str


In [94]:
# Check the actual chunk ID range

print("Minimum chunk ID:", final_chunks_df["chunk_id"].min())
print("Maximum chunk ID:", final_chunks_df["chunk_id"].max())
print("Number of unique chunk IDs:", final_chunks_df["chunk_id"].n_unique())

Minimum chunk ID: 1
Maximum chunk ID: 3596
Number of unique chunk IDs: 3596


In [95]:
# Add the two additional UNICEF sources to the source registry

additional_sources = pl.DataFrame({
    "id": [31, 32],
    "organization": ["UNICEF", "UNICEF"],
    "title": [
        "Counselling to Improve Maternal Nutrition: Considerations for Programming with Quality, Equity and Scale",
        "Evidence Review to Inform the Programme Guidance to Protect the Nutrition of Women and Adolescent Girls in Humanitarian Settings"
    ],
    "type": ["PDF", "PDF"],
    "url": [
        "https://www.unicef.org/media/114566/file",
        "https://www.unicef.org/media/160221/file/PBWG-Humanitarian-Evidence-review-final.pdf"
    ]
})

sources_df = pl.concat([
    sources_df,
    additional_sources
])

print("Total sources:", sources_df.height)

Total sources: 32


In [96]:
sources_df.filter(
    pl.col("id").is_in([31, 32])
)

id,organization,title,type,url
i64,str,str,str,str
31,"""UNICEF""","""Counselling to Improve Materna…","""PDF""","""https://www.unicef.org/media/1…"
32,"""UNICEF""","""Evidence Review to Inform the …","""PDF""","""https://www.unicef.org/media/1…"


In [97]:
source_metadata = sources_df.select([
    "id",
    "title",
    "url"
]).rename({
    "id": "source_id"
})

chunks_with_metadata = (
    chunks_with_source_id
    .drop(["title", "url"], strict=False)
    .join(
        source_metadata,
        on="source_id",
        how="left"
    )
)

final_chunks_df = chunks_with_metadata.clone()

In [98]:
for column in [
    "chunk_id",
    "organization",
    "document",
    "page_number",
    "chunk_number",
    "source_id",
    "title",
    "url",
    "text"
]:
    missing = final_chunks_df.filter(
        pl.col(column).is_null()
    ).height

    print(f"{column}: {missing} missing")

chunk_id: 0 missing
organization: 0 missing
document: 0 missing
page_number: 0 missing
chunk_number: 0 missing
source_id: 0 missing
title: 0 missing
url: 0 missing
text: 0 missing


In [99]:
from pathlib import Path

# Create the processed knowledge-base folder
processed_dir = Path("../knowledge_base/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

# Save the final chunk dataset
output_path = processed_dir / "chunks.parquet"

final_chunks_df.write_parquet(output_path)

print(f"Knowledge base saved successfully!")
print(f"File: {output_path}")
print(f"Total chunks: {final_chunks_df.height}")

Knowledge base saved successfully!
File: ..\knowledge_base\processed\chunks.parquet
Total chunks: 3596


In [100]:
# Verify the saved Parquet file

test_df = pl.read_parquet(output_path)

print("Rows:", test_df.height)
print("Columns:", test_df.columns)

Rows: 3596
Columns: ['chunk_id', 'organization', 'document', 'page_number', 'chunk_number', 'text', 'source_id', 'title', 'url']
